# 03 · Gold activity

Combine saved Silver data into one row per vehicle and UTC day. This is the data source for a separate Power BI report.

Run after `02_silver_cleaning`. Gold reads tables, not variables from Silver. Null distance means missing or unreliable data; it is not zero.

## Paths, dates and imports

In [1]:
from pathlib import Path
from datetime import date, datetime, timezone
import json
import os
import sqlite3
import uuid

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

project_dir = Path.cwd()
if not (project_dir / "weather.py").exists():
    project_dir = project_dir.parent
project_dir = Path(os.environ.get("FLEET_PROJECT_DIR", project_dir)).resolve()
source_dir = project_dir / "data" / "sources"
output_dir = project_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
start_date = date(2026, 8, 1)
end_date = date(2026, 8, 30)

# Databricks already supplies spark. Local Python needs a Delta-enabled session.
if "spark" not in globals():
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.master("local[2]").appName("fleet-activity")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.shuffle.partitions", "2"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    spark.sparkContext.setLogLevel("ERROR")
print("Report period:", start_date, "to", end_date)

:: loading settings :: url = jar:file:<repo>/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: <home>/.ivy2.5.2/cache
The jars for the packages stored in: <home>/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-844fe199-e641-4085-b059-92cb24f9699d;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central


:: resolution report :: resolve 172ms :: artifacts dl 28ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	io.delta#delta-spark_4.1_2.13;4.1.0 from central in [default]
	io.delta#delta-storage;4.1.0 from central in [default]
	io.unitycatalog#unitycatalog-client;0.4.0 from central in [default]
	org.apache.logging.log4j#log4j-api;2.25.3 from central in [default]
	org.apache.logging.log4j#log4j-core;2.25.3 from central in [default]
	org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 from central in [default]
	org.slf4j#slf4j-api;2.0.13 from central in [default]
	:: evicted modules:
	org.slf4j#slf4j-api;2.0.17 by [org.slf4j#slf4j-api;2.0.13] in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     

Report period: 2026-08-01 to 2026-08-30


## Read the saved Silver tables

In [2]:
daily = spark.read.format("delta").load(str(output_dir / "delta" / "daily_observations"))
fleet = spark.read.format("delta").load(str(output_dir / "delta" / "silver_fleet"))
weather = spark.read.format("delta").load(str(output_dir / "delta" / "silver_weather"))

## Build the reporting dates
One date per day in the reporting period. A simple loop builds the small demo calendar.

In [3]:
from datetime import timedelta

calendar_rows = []
current_day = start_date
while current_day <= end_date:
    # The trailing comma makes a one-column row.
    calendar_rows.append((current_day,))
    current_day = current_day + timedelta(days=1)
report_dates = spark.createDataFrame(calendar_rows, "event_date date")
print("Reporting days:", report_dates.count())

Reporting days: 30


## Keep missing and unknown vehicle-days
`crossJoin` creates the expected vehicle × day combinations. A left join keeps missing telemetry. The anti-join finds vehicles absent from the roster. Expected: 720 roster-days + one unknown vehicle-day.

In [4]:
expected_vehicle_days = fleet.select("vehicle_id").crossJoin(report_dates)
registered_days = expected_vehicle_days.join(daily, ["vehicle_id", "event_date"], "left")
unregistered_days = daily.join(fleet.select("vehicle_id"), "vehicle_id", "left_anti")
vehicle_days = registered_days.unionByName(unregistered_days)
vehicle_days = (vehicle_days.join(fleet, "vehicle_id", "left")
    .withColumn("registered_vehicle", F.col("depot_id").isNotNull()))
vehicle_days = (vehicle_days
    .withColumn("distance_status", F.coalesce("distance_status", F.lit("missing_telemetry")))
    .withColumn("reading_count", F.coalesce("reading_count", F.lit(0)))
    .withColumn("depot_id", F.coalesce("depot_id", F.lit("UNASSIGNED")))
    .withColumn("depot_name", F.coalesce("depot_name", F.lit("Unknown depot")))
    .withColumn("model", F.coalesce("model", F.lit("Unknown model"))))
vehicle_days.groupBy("distance_status").count().show()

+--------------------+-----+
|     distance_status|count|
+--------------------+-----+
|   missing_telemetry|    1|
|insufficient_read...|    1|
|             trusted|  716|
|           untrusted|    3|
+--------------------+-----+



## Join weather and publish Gold
Join on **depot and date**. A weather row must match many vehicles without multiplying their rows. Precipitation ≥ 1 mm is Wet; a successful null observation is Unknown.

In [5]:
gold = (vehicle_days.join(weather, ["depot_id", "event_date"], "left")
    .withColumn("weather_condition", F.when(F.col("precipitation_mm").isNull(), "Unknown")
        .when(F.col("precipitation_mm") >= 1.0, "Wet").otherwise("Dry"))
    .select("event_date", "vehicle_id", "depot_id", "depot_name", "model", "registered_vehicle",
        "reading_count", "first_reading", "last_reading", "distance_status", "observed_distance_km",
        "temperature_c", "precipitation_mm", "weather_condition"))
if gold.count() != vehicle_days.count():
    raise ValueError("Weather join multiplied or dropped vehicle-days")
if gold.select("event_date", "vehicle_id").distinct().count() != gold.count():
    raise ValueError("Gold grain is not unique")
non_trusted_distance = gold.filter(
    (F.col("distance_status") != "trusted") & F.col("observed_distance_km").isNotNull())
if non_trusted_distance.count() != 0:
    raise ValueError("Non-trusted day has a distance")
if gold.filter(F.col("observed_distance_km") < 0).count() != 0:
    raise ValueError("Negative observed distance")

gold_path = str(output_dir / "delta" / "gold_daily_vehicle_activity")
gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
gold = spark.read.format("delta").load(gold_path)
gold.createOrReplaceTempView("daily_vehicle_activity")

## Check Gold with SQL
This small summary is a validation reference for the separate Power BI report. Keep coverage beside averages so missing or unreliable days are not mistaken for zero activity.

In [6]:
activity_report = spark.sql("""
    SELECT depot_name, model, weather_condition,
           COUNT(*) AS vehicle_days,
           COUNT(observed_distance_km) AS trusted_vehicle_days,
           SUM(CASE WHEN distance_status = 'missing_telemetry' THEN 1 ELSE 0 END) AS missing_days,
           SUM(CASE WHEN distance_status = 'untrusted' THEN 1 ELSE 0 END) AS untrusted_days,
           SUM(CASE WHEN distance_status = 'insufficient_readings' THEN 1 ELSE 0 END) AS insufficient_days,
           ROUND(AVG(observed_distance_km), 2) AS mean_observed_distance_km,
           ROUND(SUM(observed_distance_km), 2) AS total_observed_distance_km
    FROM daily_vehicle_activity
    GROUP BY depot_name, model, weather_condition
    ORDER BY depot_name, model, weather_condition
""")
activity_report.show(truncate=False)

+-------------+-------------+-----------------+------------+--------------------+------------+--------------+-----------------+-------------------------+--------------------------+
|depot_name   |model        |weather_condition|vehicle_days|trusted_vehicle_days|missing_days|untrusted_days|insufficient_days|mean_observed_distance_km|total_observed_distance_km|
+-------------+-------------+-----------------+------------+--------------------+------------+--------------+-----------------+-------------------------+--------------------------+
|Chicago      |Model A      |Dry              |66          |64                  |0           |1             |1                |47.03                    |3009.61                   |
|Chicago      |Model A      |Wet              |114         |114                 |0           |0             |0                |46.34                    |5282.88                   |
|Chicago      |Model B      |Dry              |66          |65                  |0           |1

## Export Gold for Power BI

Save the Gold data as CSV for the separate Power BI report.


In [7]:
# These are small demo outputs; collecting is deliberate, not a large-data pattern.
import csv
# Format timestamps in Spark's UTC session before Python collects them.
daily_export = (gold.orderBy("event_date", "vehicle_id")
    .withColumn("first_reading", F.date_format("first_reading", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("last_reading", F.date_format("last_reading", "yyyy-MM-dd'T'HH:mm:ss'Z'")))
for name, frame in [("daily_vehicle_activity", daily_export),
                    ("activity_report", activity_report)]:
    with (output_dir / f"{name}.csv").open("w", newline="") as file:
        writer = csv.writer(file, lineterminator="\n")
        writer.writerow(frame.columns)
        writer.writerows(tuple(row) for row in frame.collect())

## Appendix: execution evidence
Read the saved tables to record counts and file hashes. Duplicate count follows the reconciliation: Bronze = accepted + quarantine + removed duplicates. This is supporting evidence, not the Power BI report.

In [8]:
bronze = spark.read.format("delta").load(str(output_dir / "delta" / "bronze_telemetry"))
quarantine = spark.read.format("delta").load(str(output_dir / "delta" / "quarantine_telemetry"))
accepted = spark.read.format("delta").load(str(output_dir / "delta" / "silver_telemetry"))
day_issues = spark.read.format("delta").load(str(output_dir / "delta" / "silver_day_issues"))
row_quarantine = quarantine.filter(F.col("quarantine_scope") == "record")
day_quarantine = quarantine.filter(F.col("quarantine_scope") == "day_hold")
exact_duplicates_removed = bronze.count() - accepted.count() - quarantine.count()
run_id = bronze.select("run_id").first()["run_id"]
database_path = source_dir / "fleet.sqlite"
weather_dir = project_dir / ("examples/weather" if os.environ.get("FLEET_OFFLINE") == "1" else "data/weather")
weather_snapshots = []
for path in sorted(weather_dir.glob("*.json")):
    snapshot = json.loads(path.read_text())
    weather_snapshots.append({"file": path.name, "request": snapshot["request"], "fetched_at": snapshot["fetched_at"]})

import hashlib
unassigned_rejects = quarantine.filter(
    F.col("vehicle_id").isNull() | (F.col("vehicle_id") == "") | F.col("event_date").isNull()).count()
record_reason_counts = {row.reason: row["count"] for row in
    row_quarantine.select(F.explode("reasons").alias("reason")).groupBy("reason").count().collect()}
day_issue_counts = {row.reason: row["count"] for row in
    day_issues.select(F.explode("reasons").alias("reason")).groupBy("reason").count().collect()}
status_counts = {row.distance_status: row["count"] for row in gold.groupBy("distance_status").count().collect()}
input_files = sorted((source_dir / "telemetry").glob("*.jsonl")) + [database_path]
input_files += sorted(weather_dir.glob("*.json"))
summary = {
    "run_id": run_id, "completed_at": datetime.now(timezone.utc).isoformat(),
    "spark_version": spark.version, "period": [str(start_date), str(end_date)],
    "bronze_rows": bronze.count(), "exact_duplicates_removed": exact_duplicates_removed,
    "quarantined_rows": quarantine.count(), "unassigned_rejects": unassigned_rejects,
    "individual_reject_rows": row_quarantine.count(), "held_rows": day_quarantine.count(),
    "individual_reject_reason_counts": record_reason_counts, "untrusted_day_reason_counts": day_issue_counts,
    "gold_rows": gold.count(), "status_counts": status_counts,
    "unregistered_vehicle_days": gold.filter(~F.col("registered_vehicle")).count(),
    "weather_rows": weather.count(), "weather_snapshots": weather_snapshots,
    "inputs": {str(path.relative_to(project_dir)): hashlib.sha256(path.read_bytes()).hexdigest() for path in input_files},
    "business_output_sha256": hashlib.sha256((output_dir / "daily_vehicle_activity.csv").read_bytes()).hexdigest(),
}
(output_dir / "run_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps({key: value for key, value in summary.items() if key not in ["inputs", "weather_snapshots"]}, indent=2))

{
  "run_id": "9626b2eb-b549-437e-9e7c-dff511f40efa",
  "completed_at": "2026-09-11T05:16:03.340329+00:00",
  "spark_version": "4.1.0",
  "period": [
    "2026-08-01",
    "2026-08-30"
  ],
  "bronze_rows": 4316,
  "exact_duplicates_removed": 1,
  "quarantined_rows": 22,
  "unassigned_rejects": 3,
  "individual_reject_rows": 4,
  "held_rows": 18,
  "individual_reject_reason_counts": {
    "invalid_odometer": 1,
    "missing_vehicle": 1,
    "invalid_timestamp": 1,
    "malformed_json": 1
  },
  "untrusted_day_reason_counts": {
    "rejected_reading_in_day": 1,
    "odometer_decreased": 1,
    "conflicting_timestamp": 1
  },
  "gold_rows": 721,
  "status_counts": {
    "missing_telemetry": 1,
    "insufficient_readings": 1,
    "trusted": 716,
    "untrusted": 3
  },
  "unregistered_vehicle_days": 1,
  "weather_rows": 60,
  "business_output_sha256": "766345118556c0b8901c8fbed8998a143c7741910c9faac6daade370e532c44c"
}


## Pipeline complete
Use `gold_daily_vehicle_activity` in Power BI. Build visuals there. No HTML, CSS or Matplotlib is required. See `POWER_BI.md` for the Gold fields and metric limits.